# Project report — Music trend detection pipeline

CS-GY 6513 Big Data | Architecture, scale, and evaluation summary.


## Architecture

1. **Ingestion (NB1–NB2)** — Raw CSV/TST to HDFS; Last.fm synthetically expanded to ~5M timestamped events; Parquet with partition pruning (`year` on events, `region` on Spotify).
2. **Batch features (NB3)** — Artist normalization, rolling play windows (1d/7d/28d), Spotify velocity and region spread, MSD audio aggregates, Billboard **forward 28-day** chart labels.
3. **Streaming (NB4)** — File source replay with watermark and sliding 7-day windows; append JSON outputs; Kafka-equivalent reader described in markdown.
4. **ML (NB5)** — Time-aware split (`week_year < 2021` train) prevents label leakage; Random Forest + scaler; AUC/F1 and feature importances.
5. **Dashboard (NB6)** — Leaderboard, per-artist trends, ROC/confusion matrix, novel positives.


## Scalability (why Spark)

- **Multi-window shuffles** — Simultaneous 7d/28d range windows over millions of events per artist require distributed shuffle and merge; single-node RAM is insufficient for wide artist skew.
- **Cross-dataset joins** — Events joined to Spotify on normalized artist keys at scale imply shuffle joins; smaller dimensions (Billboard, broadcast-eligible audio) reduce traffic.
- **Stateful streaming** — Watermarked windowed counts need durable state and backpressure; file replay exercises the same API as a Kafka source.


## Observed performance (fill on cluster)

After running NB2–NB3 on JupyterHub, record:

- Spark UI stages: shuffle read/write MB, stage latency.
- `spark.sql.shuffle.partitions=50` vs default (document any tuning).
- End-to-end wall time for feature build and training.


## Model outcomes (fill after NB5)

- AUC-ROC, F1, confusion matrix highlights.
- Top feature importances and interpretation vs proposal (momentum vs audio).
- Limitations: synthetic timestamps, Kaggle/MSD coverage gaps, label definition choices.
